## MIE360 Fall 2026 Project: Hospital Modeling and Simulation
### Deliverable 3.2: Base Model

Signatures
|Student|ID|
|-------|--|
|Gabriel Faustino|1011376823|
|Sunny Wu|1010978239|
|Benjamin Zhou|1011085772|

**Note: It is recommended to use a virtual environment (venv) before installing the required modules.** Run `python -m venv .venv` and `./.venv/Scripts/activate` before installing the required modules.

**Note: this notebook is designed to run with the locally-installed CSV data files after cloning.** Run `pip install -r ./deps/requirements.txt` before running.

In [15]:
import pandas as pd
import numpy as np
from scipy import stats
import simpy

import matplotlib.pyplot as plt
import seaborn as sns

from enum import Enum
import random
import typing
import math

In [16]:
# Time unit helper functions
def days_to_hours(d: float) -> float:
    return d * 24


def days_to_minutes(d: float) -> float:
    return d * 24 * 60


def hours_to_days(h: float) -> float:
    return h / 24


def hours_to_minutes(h: float) -> float:
    return h * 60


def minutes_to_days(m: float) -> float:
    return m / 60 / 24


def minutes_to_hours(m: float) -> float:
    return m / 60


def fraction_of_day(d: float) -> float:
    return d % 1


def hour_of_day(h: float) -> float:
    return h % 24


def minute_of_day(m: float) -> float:
    return m % (24 * 60)


def next_day(d: float) -> float:
    return math.ceil(d)


def next_day_hour(h: float) -> float:
    return days_to_hours(next_day(hours_to_days(h)))


def next_day_minute(m: float) -> float:
    return days_to_minutes(next_day(minutes_to_days(m)))

In [17]:
# Load treatment time MLEs
df_treatment_mles = pd.read_csv("./out/treatment_distribution_fits.csv", index_col="distribution_specialty")
display(df_treatment_mles)

,p,shape,loc,scale
distribution_specialty,,,,
Gamma-Gastro_Endo,0.886080,0.846696,0.0,3.986304
Gamma-Ortho,0.321779,1.014737,0.0,5.411163
Gamma-Renal,0.674105,0.512199,0.0,9.416783
Exponential-Ortho,0.379521,NaN,0.0,5.490906
Lognormal-Card,0.175105,0.986174,0.0,1.777177
Lognormal-Respi,0.280705,1.133348,0.0,1.717883


In [18]:
# Load hourly arrival Poisson MLEs
df_arrival_mles = pd.read_csv("./out/arrival_poisson_fits.csv", index_col="hour")
display(df_arrival_mles)

,lambda
hour,
1am,1.61
2am,1.25
3am,0.98
4am,0.67
5am,0.53
6am,0.49
7am,0.95
8am,1.24
9am,1.68


In [19]:
# Create *exponential* interarrival variables from data.
# We are transforming the fitted Poisson arrival MLEs (arrivals per hour) into the mean interarrival time in minutes.
hourly_arrival_distributions = []
for index, row in df_arrival_mles.iterrows():
    hourly_arrival_distributions.append(stats.expon(scale=60/row["lambda"]))

In [20]:
class TreatmentType(Enum):
    SURG = ("Surg", 0.22, stats.expon(scale=1/0.4237))
    CARD = ("Card", 0.18, stats.lognorm(s=df_treatment_mles.loc["Lognormal-Card"]["shape"], scale=df_treatment_mles.loc["Lognormal-Card"]["scale"]))
    GEN_MED = ("Gen Med", 0.16, stats.lognorm(scale=np.exp(0.921), s=0.949))
    ORTHO = ("Ortho", 0.14, stats.expon(scale=1/df_treatment_mles.loc["Exponential-Ortho"]["scale"]))
    GASTRO_ENDO = ("Gastro-Endo", 0.08, stats.gamma(a=df_treatment_mles.loc["Gamma-Gastro_Endo"]["shape"], scale=df_treatment_mles.loc["Gamma-Gastro_Endo"]["scale"]))
    ONCO = ("Onco", 0.07, stats.gamma(a=0.612, scale=9.689))
    NEURO = ("Neuro", 0.06, stats.lognorm(scale=np.exp(0.53), s=1.133))
    RENAL = ("Renal", 0.05, stats.gamma(a=df_treatment_mles.loc["Gamma-Renal"]["shape"], scale=df_treatment_mles.loc["Gamma-Renal"]["scale"]))
    RESPI = ("Respi", 0.04, stats.lognorm(s=df_treatment_mles.loc["Lognormal-Respi"]["shape"], scale=df_treatment_mles.loc["Lognormal-Respi"]["scale"]))

    def __new__(cls, name: str, proportion: float, treatment_time_rv):
        obj = object.__new__(cls)
        obj._value_ = name
        obj.type_name = name
        obj.proportion = proportion
        obj.treatment_time_rv = treatment_time_rv
        return obj

    @classmethod
    def list(cls) -> list[TreatmentType]:
        return [t for t in cls]

    @classmethod
    def weightings(cls) -> list[float]:
        return [t.proportion for t in cls]

    @classmethod
    def random_type(cls) -> TreatmentType:
        return random.choices(cls.list(), weights=cls.weightings())[0]


class Ward(object):

    def __init__(self, env: simpy.Environment,
                 ward_id: int,
                 treatment_type: TreatmentType,
                 bed_count: int,
                 physician_count: int,
                 physician_discharge_time: float,
                 physician_discharge_interval: tuple[float, float], # [3PM, 5PM]
                 cleaner_time_rv,
                 treatment_time_rv = None):
        self.env = env
        self.ward_id = ward_id
        self.treatment_type = treatment_type
        self.physician_discharge_time = physician_discharge_time
        self.physician_discharge_interval = physician_discharge_interval
        self.cleaner_time_rv = cleaner_time_rv
        self.treatment_time_rv = treatment_time_rv if treatment_time_rv else self.treatment_type.treatment_time_rv

        # Ward resources
        self.beds = simpy.Resource(env, capacity=bed_count)
        self.physicians = simpy.Resource(env, physician_count)

        # TODO trackers


    # TODO stats methods

    def get_treatment_time(self) -> float:
        return self.treatment_time_rv.ppf(random.random())

    def get_discharge_time(self) -> float:
        return self.physician_discharge_time

    def get_cleaning_time(self) -> float:
        return self.cleaner_time_rv.ppf(random.random())

    def get_discharge_interval(self) -> tuple[float, float]:
        return self.physician_discharge_interval

    def get_free_beds(self) -> int:
        return self.beds.capacity - self.beds.count

    def get_queue_length(self) -> int:
        return len(self.beds.queue)


class Hospital(object):

    def __init__(self, env: simpy.Environment,
                 simulation_time: float,
                 ward_spec: list[tuple[TreatmentType, int, typing.Any | None]],
                 physicians_per_ward: int,
                 physician_discharge_time: float,
                 physician_discharge_interval: tuple[float, float],
                 cleaner_time_rv):
        self.env = env
        self.simulation_time = simulation_time

        # Check that all wards are included in ward spec
        ward_types = set([t for t, _ in ward_spec])
        all_types = set(TreatmentType.list())
        assert all_types.issubset(ward_types), f"Ward specification does not contain treatment types {all_types.difference(ward_types)}"

        # Hospital resources
        self.wards = [Ward(env=env,
                           ward_id=ward_id,
                           treatment_type=treatment_type,
                           bed_count=bed_count,
                           physicians_per_ward=physicians_per_ward,
                           physician_discharge_time=physician_discharge_time,
                           physician_discharge_interval=physician_discharge_interval,
                           cleaner_time_rv=cleaner_time_rv,
                           treatment_time_rv=treatment_type_rv)
                      for ward_id, (treatment_type, bed_count, treatment_type_rv) in enumerate(ward_spec)]
        self.wards_by_type: dict[TreatmentType, list[Ward]] = {treatment_type: [] for treatment_type in TreatmentType.list()}
        for ward in self.wards:
            self.wards_by_type[ward.treatment_type].append(ward)

        # TODO trackers

    # TODO stats methods and helpers
    def get_wards_for_type(self, treatment_type: TreatmentType) -> list[Ward]:
        return self.wards_by_type[treatment_type]

In [21]:
# Default parameters for reference
BASE_PHYSICIANS_PER_WARD = 5
BASE_PHYSICIAN_DISCHARGE_TIME = 30 # Minutes
BASE_PHYSICIAN_ACTIIVTY_INTERVAL = (hours_to_minutes(15), hours_to_minutes(17)) # [3PM, 5PM)
BASE_CLEANING_INTERVAL = stats.triang(c=0.5, loc=30, scale=30) # T ~ Triangular(30, 45, 60)

In [22]:
# Ward allocation
def get_hospital_ward(hospital: Hospital, treatment_type: TreatmentType) -> Ward:
    wards = hospital.get_wards_for_type(treatment_type).copy()
    if len(wards) == 1:
        return wards[0]
    wards = get_wards_with_most_free_beds(wards)
    wards = get_wards_with_shortest_queues(wards)
    return wards[0] if len(wards) == 1 else random.choices(wards)[0]


def get_wards_with_most_free_beds(wards) -> list[Ward]:
    return get_best_values(wards, key_func=lambda ward: ward.get_free_beds(), select=max)


def get_wards_with_shortest_queues(wards) -> list[Ward]:
    return get_best_values(wards, key_func=lambda ward: ward.get_queue_length(), select=min)


def get_best_values(li, key_func, select) -> list[typing.Any]:
    if len(li) <= 1:
        return li
    keyed_values = {}
    for x in li:
        k = key_func(x)
        if k not in keyed_values:
            keyed_values[k] = []
        keyed_values[k].append(x)
    return keyed_values[select(keyed_values)]

In [23]:
def get_interarrival_time(env: simpy.Environment) -> float:
    """ Return the next interarrival time, in minutes, for the arrival rate
        starting at the time specified by simpy.Environment.now. """
    return hourly_arrival_distributions[hour_of_day(minutes_to_hours(env.now)) // 1].ppf(random.random())


# Process
def hospital_arrival(env, hospital):
    while True:
        interarrival_time = get_interarrival_time(env)
        yield env.timeout(interarrival_time)
        # TODO stats tracking
        treatment_type = TreatmentType.random_type()
        ward = get_hospital_ward(hospital, treatment_type)
        env.process(ward_treatment(env, ward))


def ward_treatment(env, ward):
    # TODO logging?
    bed_request = ward.beds.request()
    yield bed_request

    # TODO logging?
    treatment_time = ward.get_treatment_time()
    yield env.timeout(treatment_time)
    # TODO logging?

    yield env.process(ward_discharge(env, ward))
    # TODO logging? Patient exits here.
    yield env.process(ward_cleaning(env, ward))

    ward.beds.release(bed_request)


def ward_discharge(env, ward):
    # TODO logging?
    physician_request = ward.physicians.request()
    yield physician_request
    # Now, the physicians will handle this patient next when they can
    # Key assumption: discharge is first-come first serve;
    # the patient-processes are "waiting" but physicians will discharge
    # on basis of who finished treatment first.

    current_sim_time = env.now
    current_daytime = minute_of_day(current_sim_time)
    # Half-open interval
    discharge_interval = ward.physician_discharge_interval
    if current_daytime < discharge_interval[0]:
        yield env.timeout(discharge_interval[0] - current_daytime)
    elif current_daytime >= discharge_interval[1]:
        yield env.timeout(discharge_interval[0] + hours_to_minutes(24) - current_daytime)

    discharge_time = ward.get_discharge_time()
    yield env.timeout(discharge_time)

    ward.physicians.release(physician_request)
    # TODO logging?


def ward_cleaning(env, ward):
    # No request needed ("ample" cleaning)
    # additionally, assume cleaning runs 24/7 (unlike discharges)
    # TODO logging?
    cleaning_time = ward.get_cleaning_time()
    yield env.timeout(cleaning_time)




# TODO how to interpret this statement?
# I am thinking of just considering the assigned ward's queue as being "in ED".
# -- Gabriel
# Unavailable capacity in ward -> patient waits in ED until availability

In [24]:
# TODO simulation loops